## May Model Test on January Images

The objective of this script is to test a YOLO model trained on May images on new sets of January test images

First, let's import the necessary libraries.

In [2]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.5 MB/s eta 0:00:00


In [3]:
import os
import yaml
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### 1. Testing Preparation

We first mount our drive to point to the folder where the testing images are located.

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


Next, we load the trained model

<div class="alert alert-block alert-info">
    
<b>Note:</b> YOLOv8 automatically saves the model on training. The saved model can be found in this path where the training script is located. *runs/detect/train/exp*/weights/*

The model is automatically named as *"best.pt"*

</div>

In [5]:
model_path = os.path.join("/content/drive/MyDrive/Drone_images/MAY_Training/runs/detect/train/weights", "best.pt")
may_model = YOLO(model_path)

Next we define the path of the test images including the labels (Pascal VOC .xml labels and .txt class files)

In [6]:
test_images = "/content/drive/MyDrive/Drone_images/JAN_Training/images/test"   # folder with test images
test_labels = "/content/drive/MyDrive/Drone_images/JAN_Training/labels/test"   # folder with YOLO .txt labels
xml_labels  = "/content/drive/MyDrive/Drone_images/JAN_Training/labels/test"   # folder with XML files

### 2. Testing the LWIR Images

We make predictions using the trained model

In [7]:
pred_results = may_model.predict(source = test_images, imgsz = 640, save = True)


image 1/174 /content/drive/MyDrive/Drone_images/JAN_Training/images/test/jan_afternoon_0_39.jpg: 512x640 1 ap_plastic, 4 at_plastics, 68.9ms
image 2/174 /content/drive/MyDrive/Drone_images/JAN_Training/images/test/jan_afternoon_0_47.jpg: 512x640 2 ap_metals, 2 at_plastics, 57.4ms
image 3/174 /content/drive/MyDrive/Drone_images/JAN_Training/images/test/jan_afternoon_0_5.jpg: 512x640 3 ap_plastics, 3 at_plastics, 18.9ms
image 4/174 /content/drive/MyDrive/Drone_images/JAN_Training/images/test/jan_afternoon_0_7.jpg: 512x640 (no detections), 7.7ms
image 5/174 /content/drive/MyDrive/Drone_images/JAN_Training/images/test/jan_afternoon_0_8.jpg: 512x640 1 ap_plastic, 2 at_metals, 3 at_plastics, 5.9ms
image 6/174 /content/drive/MyDrive/Drone_images/JAN_Training/images/test/jan_afternoon_100_10.jpg: 512x640 2 at_plastics, 5.9ms
image 7/174 /content/drive/MyDrive/Drone_images/JAN_Training/images/test/jan_afternoon_100_19.jpg: 512x640 2 at_plastics, 5.9ms
image 8/174 /content/drive/MyDrive/Drone_i

### 3. Model Evaluation

First, we create aand save structured configuration YAML file that will help us in evaluating the model.

In [8]:
class_names = ['ap_metal', 'ap_plastic', 'at_metal', 'at_plastic']

In [9]:
data = {
    "path":"/content/drive/MyDrive/Drone_images/JAN_Training",
    "train": os.path.join("/content/drive/MyDrive/Drone_images/JAN_Training/images/train"),
    "val": os.path.join("/content/drive/MyDrive/Drone_images/JAN_Training/images/val"),
    "test": os.path.join("/content/drive/MyDrive/Drone_images/JAN_Training/images/test"),
    "names": class_names,
}

yaml_dir = "/content/drive/MyDrive/Drone_images/MAY_Testing"
yaml_path = os.path.join(yaml_dir, "eval.yaml")
with open(yaml_path, "w") as f:

    yaml.dump(data, f, default_flow_style = False)

We finally evaluate the performance of our model by printing and saving the precision, recall, 50% and 90% mean Average Precision (mAP) scores.

In [10]:
results = may_model.val(data = yaml_path, split = "test",
    imgsz = 640, batch = 16, save_json = True, plots = True)

Ultralytics 8.3.203 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 4.2±8.2 ms, read: 43.9±38.6 MB/s, size: 142.8 KB)
val: Scanning /content/drive/MyDrive/Drone_images/JAN_Training/labels/test.cache... 174 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 174/174 46.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.1it/s 5.3s
                   all        174       1264       0.36       0.26      0.236       0.12
              ap_metal         74         92      0.145       0.12     0.0746     0.0354
            ap_plastic         96        215      0.365      0.167      0.167     0.0719
              at_metal         88        122      0.184      0.164     0.0811     0.0466
            at_plastic        167        835      0.747       0.59      0.621      0.327
Speed: 2.2ms preprocess, 4.9ms inference, 0.0ms loss, 3.1ms postprocess per image
Saving /content/ru

We save the model evaluation results for future use.

In [11]:
!cp -r /content/runs/detect/val /content/drive/MyDrive/Drone_images/MAY_Testing/may_model_predictions_on_jan/

And then extract the individual metrics before converting to dataframe and eventually saving the results as a csv file.

In [12]:
precision = results.box.p
recall = results.box.r
ap50 = results.box.ap50
ap5095 = results.box.ap
class_names = results.names

In [13]:
df = pd.DataFrame({
    "class": [class_names[i] for i in range(len(precision))],
    "precision": precision,
    "recall": recall,
    "map50": ap50,
    "map50_95": ap5095,
    "model": ["May on Jan Images"] * len(precision)
    })

RESULTS = "/content/drive/MyDrive/Drone_images/MAY_Testing"
output_file = os.path.join(RESULTS, "may_model_on_jan_images_metrics.csv")
df.to_csv(output_file, index=False)